# Hyperparameter Tuning with GridSearchCV and RandomizedSearchCV

## Can we do better?

After training a model and evaluating its performance, the next logical question is: **Can we do better?**

Most machine learning models have **hyperparameters** — configuration settings that control model behavior but are not learned from the data. They are set by the practitioner, before training begins, and they can make an enormous difference.

**The difference between a working model and a strong model is often just a hyperparameter configuration.**

## Section 1: Understanding Hyperparameters and Their Impact

### What Are Hyperparameters?

Hyperparameters are settings defined **before training** that govern how the model learns. They differ fundamentally from model parameters:

| Type | Example | Learned from Data? | Set By |
|------|---------|-------------------|--------|
| Model Parameter | Regression coefficients | Yes — optimized during training | The algorithm |
| Hyperparameter | K in KNN | No — fixed before training | The practitioner |
| Hyperparameter | max_depth in Decision Tree | No — fixed before training | The practitioner |

### Examples of Key Hyperparameters

- **K in KNN** — controls neighborhood size and the bias–variance trade-off
- **max_depth in Decision Trees** — controls how complex the learned rules can become  
- **C in Logistic Regression** — controls regularization strength
- **n_estimators in Random Forest** — controls ensemble size

### The Impact of Hyperparameter Choice

In practice, the difference between a default configuration and a tuned configuration is often **5–15% in the metric that matters** — enough to determine whether a model gets deployed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All required libraries imported successfully")

## Demonstration: Why Hyperparameter Tuning Is Necessary

Let's see how different `max_depth` values affect a Decision Tree's performance:

In [ ]:
# Create a sample dataset
X, y = make_classification(
    n_samples=1000, 
    n_features=20, 
    n_informative=15,
    n_redundant=5,
    random_state=42,
    class_sep=1.0
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Demonstrate the effect of max_depth on Decision Trees
depths = [2, 4, 6, 8, 10, None]
train_accuracies = []
test_accuracies = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

# Create results DataFrame
results_depths = pd.DataFrame({
    'max_depth': [str(d) if d else 'None' for d in depths],
    'Train Accuracy': train_accuracies,
    'Test Accuracy': test_accuracies,
    'Diagnosis': [
        'High bias (underfitting)',
        'Good balance',
        'Entering overfitting',
        'Overfitting',
        'Overfitting',
        'Extreme overfitting'
    ]
})

print("\nDecision Tree: Effect of max_depth")
print("="*70)
print(results_depths.to_string(index=False))
print("\n✓ Notice how test accuracy peaks at a specific depth, then declines")

## Section 2: Setting Up Data and Pipelines Correctly

### Why Pipelines Matter

**Critical Rule:** Always wrap preprocessing and model training inside a Pipeline to prevent data leakage during cross-validation.

If you scale features outside the pipeline before GridSearchCV:
- The scaler sees the entire training set during fitting
- Validation-fold features are scaled using statistics that **include** those same validation samples
- This is **data leakage** and inflates all CV scores

**Solution:** Wrap scaling inside the pipeline so each fold's validation set remains isolated.

In [ ]:
# ✓ CORRECT: Split BEFORE any tuning
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size:     {X_test.shape[0]}")
print("\n✓ Test set is now locked away and will NOT be used during tuning")

# ✓ CORRECT: Preprocessing inside Pipeline
pipeline_correct = Pipeline([
    ("scaler", StandardScaler()),           # Preprocessing step
    ("knn", KNeighborsClassifier())         # Model step
])

print("\n✓ Pipeline structure:")
print(pipeline_correct)
print("\nPreprocessing INSIDE the pipeline prevents data leakage during CV")

## Section 3: Implementing GridSearchCV with KNN

### How GridSearchCV Works

GridSearchCV performs five steps:

1. **Define a grid** — which hyperparameters to search and which values to try
2. **Generate all combinations** — creates Cartesian product of the grid (e.g., 20 K values × 2 weight types = 40 combinations)
3. **Cross-validate each combination** — for each combination, run 5-fold CV on training set; compute mean validation score
4. **Select the best combination** — the one with highest mean CV score
5. **Refit on full training data** — the best configuration is trained on all available training data

Step 5 is often overlooked: after identifying best hyperparameters, GridSearchCV **automatically retrains** on the entire training set, not just one fold.

In [ ]:
# Define the parameter grid
param_grid_knn = {
    "knn__n_neighbors": range(1, 21),          # K values: 1 to 20
    "knn__weights": ["uniform", "distance"]    # Two weight options
}

print("Parameter Grid for KNN:")
print(f"  - n_neighbors: {list(range(1, 21))}")
print(f"  - weights: ['uniform', 'distance']")
print(f"\nGrid size: 20 × 2 = 40 combinations")
print(f"With 5-fold CV: 40 × 5 = 200 model fits\n")

# Create GridSearchCV
grid_knn = GridSearchCV(
    pipeline_correct,
    param_grid_knn,
    cv=5,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1          # Use all available CPU cores
)

print("Running GridSearchCV on training set (this may take 10-15 seconds)...")
grid_knn.fit(X_train, y_train)

print("\n✓ GridSearchCV complete!")
print(f"\nBest Parameters:  {grid_knn.best_params_}")
print(f"Best CV Accuracy: {grid_knn.best_score_:.4f}")
print(f"Best Model:       {grid_knn.best_estimator_}")

In [ ]:
# ✓ CRITICAL: Evaluate on test set EXACTLY ONCE, only AFTER tuning is complete
best_model_knn = grid_knn.best_estimator_
y_pred_knn = best_model_knn.predict(X_test)
test_accuracy_knn = accuracy_score(y_test, y_pred_knn)

print("="*70)
print("TEST SET EVALUATION (After tuning is complete)")
print("="*70)
print(f"\nTest Accuracy: {test_accuracy_knn:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

print("\n✓ Note: Test accuracy is evaluated EXACTLY ONCE")
print("  The test set has been held out throughout the entire search process")

## Section 4: Interpreting and Visualizing Search Results

### Accessing the Complete Results

All search results are stored in `grid.cv_results_` — a dictionary containing every combination and its cross-validation scores:

In [ ]:
# Extract results into a DataFrame for analysis
results_df = pd.DataFrame(grid_knn.cv_results_)[[
    "param_knn__n_neighbors",
    "param_knn__weights",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]]

# Rename for clarity
results_df.columns = [
    "K", "Weights", 
    "Mean Train Accuracy", "Mean CV Accuracy", 
    "CV Std", "Rank"
]

print("Top 10 Configurations by CV Accuracy")
print("="*90)
print(results_df.sort_values("Rank").head(10).to_string(index=False))

print("\n\nKey Insights:")
print("-" * 90)
best_row = results_df.sort_values("Rank").iloc[0]
print(f"✓ Best Configuration: K={int(best_row['K'])}, Weights={best_row['Weights']}")
print(f"  Mean CV Accuracy: {best_row['Mean CV Accuracy']:.4f} ± {best_row['CV Std']:.4f}")
print(f"  Train Accuracy:   {best_row['Mean Train Accuracy']:.4f}")

train_cv_gap = best_row['Mean Train Accuracy'] - best_row['Mean CV Accuracy']
print(f"  Train-CV Gap:     {train_cv_gap:.4f}")

if train_cv_gap > 0.10:
    print(f"  ⚠ Large train-CV gap suggests possible overfitting")

In [ ]:
# Visualize the results
uniform_mask = results_df["Weights"] == "uniform"
distance_mask = results_df["Weights"] == "distance"

k_values = results_df[uniform_mask]["K"].astype(int).values
uniform_train = results_df[uniform_mask]["Mean Train Accuracy"].values
uniform_cv = results_df[uniform_mask]["Mean CV Accuracy"].values
uniform_std = results_df[uniform_mask]["CV Std"].values
distance_cv = results_df[distance_mask]["Mean CV Accuracy"].values

fig, ax = plt.subplots(figsize=(12, 6))

# Plot train and CV scores
ax.plot(k_values, uniform_train, 
        label="Train (uniform)", linestyle="--", marker="o", 
        color="C0", alpha=0.7, linewidth=2, markersize=6)
ax.plot(k_values, uniform_cv, 
        label="CV (uniform)", marker="o", 
        color="C1", linewidth=2, markersize=6)
ax.plot(k_values, distance_cv, 
        label="CV (distance)", marker="s", linestyle=":", 
        color="C2", linewidth=2, markersize=6)

# Add confidence interval
ax.fill_between(
    k_values,
    uniform_cv - uniform_std,
    uniform_cv + uniform_std,
    alpha=0.2, color="C1", label="CV ± 1 Std"
)

# Annotations
best_k = int(results_df.sort_values("Rank").iloc[0]["K"])
best_cv = results_df.sort_values("Rank").iloc[0]["Mean CV Accuracy"]
ax.axvline(x=best_k, color="red", linestyle=":", alpha=0.5, linewidth=2)
ax.plot(best_k, best_cv, marker="*", markersize=20, color="red", 
        label=f"Best (K={best_k})")

# Regions
ax.axvspan(0.5, 3, alpha=0.1, color="purple", label="Overfitting Region")
ax.axvspan(best_k-1, best_k+1, alpha=0.1, color="green", label="Optimal Region")
ax.axvspan(18, 20.5, alpha=0.1, color="orange", label="Underfitting Region")

ax.set_xlabel("K (Number of Neighbors)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("GridSearchCV: K vs. Cross-Validation Accuracy (KNN)", fontsize=14, fontweight='bold')
ax.legend(loc="best", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.7, 1.0])

plt.tight_layout()
plt.show()

print("\nVisualization Interpretation:")
print("-" * 90)
print("• At small K (1-3):  High train accuracy, low CV accuracy → OVERFITTING")
print("• At optimal K:      Train and CV scores converge → GOOD GENERALIZATION")
print("• At large K (18-20): Both scores decline → UNDERFITTING")

## Section 5: GridSearchCV with Decision Trees

Decision Trees have multiple important hyperparameters. Let's tune them systematically:

In [ ]:
# Multi-parameter grid for Decision Trees
param_grid_dt = {
    "max_depth":        [2, 4, 6, 8, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "criterion":        ["gini", "entropy"]
}

# Calculate grid size
n_combinations = 6 * 4 * 2
cv_folds = 5
total_fits = n_combinations * cv_folds

print(f"Decision Tree Parameter Grid")
print("="*70)
print(f"  max_depth:        {[2, 4, 6, 8, 10, None]} (6 values)")
print(f"  min_samples_leaf: {[1, 5, 10, 20]} (4 values)")
print(f"  criterion:        {['gini', 'entropy']} (2 values)")
print(f"\nGrid Computation:")
print(f"  Combinations: {n_combinations}")
print(f"  CV Folds:     {cv_folds}")
print(f"  Total Fits:   {total_fits}")

# Run GridSearchCV with F1 scoring (better for imbalanced classification)
print(f"\nRunning GridSearchCV with {total_fits} model fits...")
print("(This may take 20-30 seconds)\n")

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=5,
    scoring="f1_weighted",  # Use F1 to handle potential class imbalance
    return_train_score=True,
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("✓ GridSearchCV complete!")
print(f"\nBest Parameters: {grid_dt.best_params_}")
print(f"Best CV F1 Score: {grid_dt.best_score_:.4f}")

# Test set evaluation
y_pred_dt = grid_dt.best_estimator_.predict(X_test)
test_f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
print(f"Test F1 Score:    {test_f1_dt:.4f}")

print("\n" + "="*70)
print(classification_report(y_test, y_pred_dt))

## Section 6: Choosing the Right Scoring Metric

### Why the Scoring Metric Matters as Much as the Grid

GridSearchCV selects hyperparameters by **maximizing the scoring metric**. Choosing the wrong metric means optimizing for the wrong objective.

| Problem Type | Dataset Balance | Recommended Scoring |
|---|---|---|
| Classification | Balanced | `accuracy` |
| Classification | Imbalanced | `f1` or `roc_auc` |
| Classification, FN costly | Any | `recall` |
| Classification, FP costly | Any | `precision` |
| Regression | Any | `neg_mean_squared_error` or `r2` |

**Important**: For regression metrics (MSE, MAE), scikit-learn returns negated values (e.g., `neg_mean_squared_error`) because it uses "higher is better" convention. When reporting, negate it back.

In [ ]:
# Demonstrate how different metrics lead to different best hyperparameters
param_grid_simple = {
    "knn__n_neighbors": range(1, 16)
}

scoring_metrics = ["accuracy", "f1_weighted", "roc_auc", "recall_weighted"]
results_by_metric = {}

print("Comparing different scoring metrics on the same KNN pipeline\n")
print("="*80)

for metric in scoring_metrics:
    try:
        grid_metric = GridSearchCV(
            pipeline_correct,
            param_grid_simple,
            cv=5,
            scoring=metric,
            n_jobs=-1
        )
        grid_metric.fit(X_train, y_train)
        
        results_by_metric[metric] = {
            'best_params': grid_metric.best_params_['knn__n_neighbors'],
            'best_cv_score': grid_metric.best_score_,
            'test_score': grid_metric.score(X_test, y_test)
        }
        
        print(f"\nMetric: {metric}")
        print(f"  Best K: {grid_metric.best_params_['knn__n_neighbors']}")
        print(f"  Best CV Score: {grid_metric.best_score_:.4f}")
        print(f"  Test Score: {grid_metric.score(X_test, y_test):.4f}")
        
    except Exception as e:
        print(f"\n⚠ Metric '{metric}' not available: {e}")

print("\n" + "="*80)
print("\n✓ Notice how the best K value changes depending on the scoring metric!")
print("  The choice of metric DIRECTLY affects hyperparameter selection.")
print("  This is why aligning the metric with business objectives is critical.")

## Section 7: Avoiding Data Leakage During Tuning

### The Most Important Rule (and Most Commonly Violated)

**Never let test-set information influence hyperparameter selection.**

### The Correct Workflow

```
All Data
┌─────────────────────────────────────┐
│ Training Set        │ Test Set      │
│                     │               │
│ • GridSearchCV      │ • Evaluated   │
│   runs here         │   once here   │
│ • CV splits         │   ONLY after  │
│   training only     │   tuning      │
└─────────────────────────────────────┘
```

### Why This Matters

If you tune on the test set and evaluate on the same test set:
- Reported performance reflects how well the model is optimized for **that specific test sample**
- **Not** how well it generalizes to new, unseen data
- The published metric is overly optimistic
- The model will perform worse in production than expected

### Common Data Leakage Mistakes

1. **❌ Running GridSearchCV on full dataset before splitting** → test labels influence selection
2. **❌ Looking at test accuracy for different hyperparameters and choosing based on that** → direct leakage
3. **❌ Tuning, evaluating on test, tuning again based on test results** → iterative leakage
4. **❌ Scaling features outside the pipeline** → scaler sees validation fold samples